# Module 11: Lead and Lag Between Two Series

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

"Does call volume move before use of force does?" is a reasonable question and
an easy one to answer wrongly.

Two public safety series almost always correlate, because both rise in summer
and both drift with the same long run forces. Correlating them as they stand
measures the calendar they share, not any link between them.

**About 20 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
from pathlib import Path

import numpy as np
import pandas as pd

GITHUB = "https://raw.githubusercontent.com/OWNER/REPO/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

print("reading from:", BASE)

monthly = pd.read_csv(BASE + "agency_monthly.csv")
final = monthly[monthly["provisional"] == 0]        # never fit on unfinished months


def series(agency_id, column="n_uof"):
    """One agency's monthly series, indexed by date."""
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series(d[column].values, dtype=float,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


def rate(agency_id):
    d = final[final["agency_id"] == agency_id].sort_values("year_month")
    return pd.Series((100 * d["n_uof"] / d["n_arrests"]).values,
                     index=pd.PeriodIndex(d["year_month"], freq="M").to_timestamp())


grandview = series("A012")
print(f"{len(grandview)} months, {grandview.index.min():%Y-%m} to {grandview.index.max():%Y-%m}")

calls = series("A012", "total_cfs")
arrests = series("A012", "n_arrests")
uof = series("A012")
print(f"three series, {len(uof)} months each")

## 2. The cross correlation function

Shift one series against the other and correlate at each shift. A negative lag
means the first series leads.

In [ ]:
def ccf(x, y, maxlag=12):
    """Correlation of x shifted by k against y. Negative k means x leads."""
    n = len(x)
    out = {}
    for k in range(-maxlag, maxlag + 1):
        if k < 0:
            out[k] = float(np.corrcoef(x[-k:], y[:n + k])[0, 1])
        elif k == 0:
            out[k] = float(np.corrcoef(x, y)[0, 1])
        else:
            out[k] = float(np.corrcoef(x[:n - k], y[k:])[0, 1])
    return pd.Series(out)


band = 1.96 / np.sqrt(len(uof))
print(f"anything inside plus or minus {band:.2f} is indistinguishable from zero")

## 3. What it looks like on the raw series

In [ ]:
raw = ccf(calls, uof)
print("calls for service against use of force, raw")
for k in [-12, -6, -1, 0, 1, 6, 12]:
    print(f"  lag {k:+3d}: {raw[k]:+.2f}")
print(f"\nlargest value is at lag {raw.idxmax():+d}, at {raw.max():+.2f}")

Read this literally and it says calls for service **twelve months ago** are the
best predictor of use of force today, better than calls this month.

That is nonsense, and the shape gives it away: the values oscillate, positive
near lag 0, strongly negative near lag 6, positive again at lag 12. It is a
wave with a twelve month period. Both series carry the same season, so shifting
one by six months lines its summer up against the other's winter.

**A cross correlation computed on seasonal series measures the calendar.**

## 4. Remove what the two series share, then look again

Take the remainder from an STL decomposition of each series. What is left is
the month to month movement that the trend and the season do not explain, for
each series separately.

In [ ]:
from statsmodels.tsa.seasonal import STL

def remainder(s):
    return np.exp(STL(np.log(s), period=12, robust=True).fit().resid)

clean_calls, clean_arrests, clean_uof = (remainder(calls), remainder(arrests),
                                         remainder(uof))

out = pd.DataFrame({
    "calls to use of force": ccf(clean_calls, clean_uof),
    "arrests to use of force": ccf(clean_arrests, clean_uof),
}).round(2)
out["outside the band"] = np.where(
    (out.abs() > band).any(axis=1),
    out.abs().idxmax(axis=1).where((out.abs() > band).any(axis=1), ""), "")
out.loc[-4:4]

## 5. The two answers

**Calls for service tell you almost nothing.** At lag 0 the correlation is about
0.12, inside the noise band. A few scattered lags poke outside it, which is what
twenty five simultaneous tests produce by chance.

**Arrests tell you a great deal, at lag 0 and nowhere else.**

In [ ]:
for name, x in [("calls for service", clean_calls), ("arrests", clean_arrests)]:
    c = ccf(x, clean_uof)
    big = {k: round(v, 2) for k, v in c.items() if abs(v) > band}
    print(f"{name:20s} lag 0: {c[0]:+.2f}   outside the band: {big}")

Arrests at lag 0 reaches +0.44 and is the **only** value in the whole function
that clears the band. That is the shape of a real, immediate relationship, and
it is what the dataset was built to contain: use of force arises out of arrests
in the same month.

The pair of results together is what makes the exercise convincing. The method
found nothing where there is nothing, and found one clean thing where there is
one. A method that flags something everywhere is not detecting, it is decorating.

## 6. Why calls do not reach through to use of force

Calls lead to arrests, and arrests lead to use of force, but each step adds its
own variation. By the time you get from calls to use of force, two layers of
noise sit between them and the month to month signal has mostly washed out.

In [ ]:
chain = pd.Series({
    "calls to arrests": ccf(clean_calls, clean_arrests)[0],
    "arrests to use of force": ccf(clean_arrests, clean_uof)[0],
    "calls to use of force": ccf(clean_calls, clean_uof)[0],
}).round(2)
chain

Each link is real and moderate. The product of the two is weak. That is an
ordinary feature of chained relationships and a good reason to measure the step
you care about rather than the one that is easiest to get data for.

## 7. What a lead would look like, and what it would not prove

If a genuine lead existed, the function would peak at a non zero lag and be
small at the others. Two cautions apply even then.

**A lead is not a cause.** Something that moves first may be reacting to the
same driver a little sooner.

**Testing 25 lags means expecting about one false positive.** Decide which lags
are plausible before looking, or adjust for how many you tested.

## Exercise

Run the same analysis on Tarnbridge. Does its arrests to use of force link
look like Ashfell's?

In [ ]:
# Fill in the blank, then run.
AGENCY = None              # try "A002"

if AGENCY:
    a = remainder(series(AGENCY, "n_arrests"))
    u = remainder(series(AGENCY))
    b = 1.96 / np.sqrt(len(u))
    c = ccf(a, u)
    print(f"{AGENCY}: noise band plus or minus {b:.2f}")
    print(f"  lag 0: {c[0]:+.2f}")
    print("  outside the band:",
          {k: round(v, 2) for k, v in c.items() if abs(v) > b})
else:
    print("Set AGENCY above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
AGENCY = "A002"
```

No. Tarnbridge returns about **+0.08 at lag 0, inside the noise band, with
nothing anywhere in the function outside it.** On this evidence alone you would
conclude there is no relationship between arrests and use of force at Cedar
Falls.

That conclusion would be wrong, and we can be certain it is wrong, because this
dataset was built with use of force generated from arrests in the same month
**identically for every agency**. The relationship at Tarnbridge is exactly as
real as the one at Ashfell.

What differs is the amount of evidence. Ashfell records about 100 incidents a
month and Tarnbridge about 28, so a much larger share of Tarnbridge' month to
month movement is counting noise, and the signal is buried under it. This is
[Module 4](Module_04_Why_Small_Agencies_Look_Volatile.md) appearing in a new
place: the same real effect becomes undetectable as the agency gets smaller.

Excluding the June 2021 unrest month changes the answer by about 0.01, so the
outlier is not the reason. Sample size is.

**The practical rule: a cross correlation inside the band means "not detectable
here", never "not there".** If the relationship matters, pool agencies or pool
years until there is enough data to see it.

</details>

---

**Next:** [Module 12, Building a Peer Benchmark Series](Module_12_Building_A_Peer_Benchmark_Series.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*